<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания - 8
### Выполнил: Железняк Дамир Николаевич
### Группа: ПКТб-25-1


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----


Развитие системы управления кредитными картами (Спринт 3). Проект включает базовый класс CreditCard и производные классы (GoldCard, CorporateCard, VIPCorporateCard), демонстрирующие инкапсуляцию, простое, сложное и множественное наследование (через интерфейсы).

### Дополнительное задание:
К существующей структуре добавлены новые атрибуты (Currency, DailyLimit) и методы (SetDailyLimit).
Реализованы два вида полиморфизма:   
- Перекрытие (Override): Классы-наследники переопределяют поведение базовых методов GetInfo(), CheckBalance() и Pay().
- Перегрузка (Overload): Метод Pay имеет несколько сигнатур — Pay(decimal amount) для обычной оплаты и Pay(decimal amount, string category) для оплаты с указанием категории и расчетом дополнительного кэшбэка.

<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [3]:
using System;
using System.Collections.Generic;

// 1. ИСПОЛНЯЕМЫЙ КОД (Top-level statements)

// --- Воспроизведение вывода из прошлого спринта ---
GoldCard card1 = new GoldCard("1111-2222-3333", "Иван Иванов", "12.28", 50000m);
CorporateCard card2 = new CorporateCard("7777-8888-9999", "Петр Петров", "10.29", 500000m, "ООО Инновации");

Console.WriteLine(card1.GetInfo());
card1.CheckBalance();
Console.WriteLine();

Console.WriteLine(card2.GetInfo());
card2.CheckBalance();
Console.WriteLine();

card2.Transfer(card1, 25000m);
Console.WriteLine();

card1.CheckBalance();
card2.CheckBalance();
Console.WriteLine(new string('-', 50));
Console.WriteLine();

Console.WriteLine("[Демонстрация дополнительного задания]");
VIPCorporateCard vipCard = new VIPCorporateCard("5555-6666-7777", "Анна Смирнова", "11.30", 150000m, "ООО Инновации", "Михаил");
Console.WriteLine(vipCard.GetInfo());
vipCard.CallManager(); 

card1.Pay(5000m); 
card2.BlockCard(); 
card2.Pay(1000m);  


// --- Демонстрация 3-го спринта (Перегрузка методов и Generic классы) ---
Console.WriteLine("\n==================================================");

// 1. Перегрузка методов (Overload)
Console.WriteLine("\n--- Перегрузка метода Pay ---");
// Вызов перегруженной версии с категорией
card1.Pay(3000m, "Рестораны"); 
card1.Pay(1500m, "Аптеки");    

// Демонстрация новых атрибутов
card1.SetDailyLimit(10000m);

// 2. Generic класс
Console.WriteLine("\n--- Работа Generic класса CardManager<T> ---");
// Создаем хранилище, которое принимает ТОЛЬКО объекты, унаследованные от CreditCard
CardManager<CreditCard> manager = new CardManager<CreditCard>();

manager.AddCard(card1);
manager.AddCard(card2);
manager.AddCard(vipCard);

manager.ShowAllCardsInfo();

// 2. ИНТЕРФЕЙСЫ, GENERIC И ОБЪЯВЛЕНИЕ КЛАССОВ

// Интерфейс для множественного наследования
public interface IBonusSystem
{
    int BonusPoints { get; set; }
    void AddBonus(decimal amount);
}

// GENERIC КЛАСС (Обобщенный класс) для управления картами
// where T : CreditCard означает, что тип T обязан быть CreditCard или его наследником
public class CardManager<T> where T : CreditCard
{
    private List<T> _cards = new List<T>();

    public void AddCard(T card)
    {
        _cards.Add(card);
    }

    public void ShowAllCardsInfo()
    {
        Console.WriteLine($"В менеджере сохранено карт: {_cards.Count}");
        foreach (var card in _cards)
        {
            // Поскольку T ограничен CreditCard, мы можем безопасно вызывать GetInfo()
            Console.WriteLine($" - {card.GetInfo()}");
        }
    }
}

// Базовый класс
public class CreditCard
{
    private decimal _balance;

    public string CardNumber { get; set; }
    public string HolderName { get; set; }
    public string ExpiryDate { get; set; }
    public bool IsBlocked { get; set; } = false;
    
    // Новые атрибуты (Спринт 3)
    public string Currency { get; set; } = "руб.";
    public decimal DailyLimit { get; set; } = 50000m;

    public decimal Balance
    {
        get { return _balance; }
        set { _balance = value >= 0 ? value : 0; }
    }

    public CreditCard(string cardNumber, string holderName, string expiryDate, decimal initialBalance)
    {
        CardNumber = cardNumber;
        HolderName = holderName;
        ExpiryDate = expiryDate;
        Balance = initialBalance;
    }

    // Новый метод (Спринт 3)
    public void SetDailyLimit(decimal limit)
    {
        DailyLimit = limit;
        Console.WriteLine($"[Лимит] Для карты {CardNumber} установлен дневной лимит: {DailyLimit} {Currency}");
    }

    public virtual string GetInfo()
    {
        return $"Карта: {CardNumber} | Владелец: {HolderName} | Срок: {ExpiryDate}";
    }

    public virtual void CheckBalance()
    {
        Console.WriteLine($"Текущий баланс: {Math.Round(Balance)} {Currency}");
    }

    public void Transfer(CreditCard targetCard, decimal amount)
    {
        if (!IsBlocked && !targetCard.IsBlocked && amount > 0 && Balance >= amount)
        {
            Balance -= amount;
            targetCard.Balance += amount;
            Console.WriteLine($"Перевод {Math.Round(amount)} {Currency} с карты {CardNumber} на карту {targetCard.CardNumber} выполнен успешно.");
        }
        else
        {
            Console.WriteLine($"Ошибка перевода. Проверьте баланс или статус блокировки карт.");
        }
    }

    // ПЕРЕГРУЗКА МЕТОДА №1: Обычная оплата (принимает 1 параметр)
    public virtual void Pay(decimal amount)
    {
        if (IsBlocked)
        {
            Console.WriteLine($"[Отказ] Карта {CardNumber} заблокирована.");
            return;
        }

        if (amount > 0 && Balance >= amount)
        {
            Balance -= amount;
            Console.WriteLine($"Оплата {Math.Round(amount)} {Currency} успешно проведена.");
        }
    }

    // ПЕРЕГРУЗКА МЕТОДА №2 (Спринт 3): Оплата с указанием категории (принимает 2 параметра)
    public virtual void Pay(decimal amount, string category)
    {
        if (IsBlocked)
        {
            Console.WriteLine($"[Отказ] Карта {CardNumber} заблокирована.");
            return;
        }

        if (amount > 0 && Balance >= amount)
        {
            Balance -= amount;
            Console.WriteLine($"Оплата {Math.Round(amount)} {Currency} в категории '{category}' успешно проведена.");
        }
    }

    public void BlockCard()
    {
        IsBlocked = true;
        Console.WriteLine($"[Безопасность] Карта {CardNumber} была заблокирована.");
    }
}

// Производный класс 1 (Множественное наследование)
public class GoldCard : CreditCard, IBonusSystem
{
    public int BonusPoints { get; set; } = 0;

    public GoldCard(string cardNumber, string holderName, string expiryDate, decimal initialBalance) 
        : base(cardNumber, holderName, expiryDate, initialBalance) { }

    public void AddBonus(decimal amount)
    {
        int earned = (int)(amount / 100);
        BonusPoints += earned;
        Console.WriteLine($"[Бонус] Начислено {earned} баллов. Баланс баллов: {BonusPoints}.");
    }

    // Перекрытие первой версии Pay
    public override void Pay(decimal amount)
    {
        if (!IsBlocked && Balance >= amount)
        {
            base.Pay(amount);
            AddBonus(amount); 
        }
        else
        {
            base.Pay(amount); 
        }
    }

    // Перекрытие второй (перегруженной) версии Pay
    public override void Pay(decimal amount, string category)
    {
        if (!IsBlocked && Balance >= amount)
        {
            base.Pay(amount, category);
            AddBonus(amount);
            
            // Специфичная логика полиморфизма для категории
            if (category == "Рестораны")
            {
                Console.WriteLine($" -> [Кэшбэк] Применен повышенный кэшбэк для категории '{category}'!");
            }
        }
        else
        {
            base.Pay(amount, category);
        }
    }
}

// Производный класс 2
public class CorporateCard : CreditCard
{
    public string Company { get; set; }

    public CorporateCard(string cardNumber, string holderName, string expiryDate, decimal initialBalance, string company) 
        : base(cardNumber, holderName, expiryDate, initialBalance)
    {
        Company = company;
    }

    public override string GetInfo()
    {
        return base.GetInfo() + $" | Компания: {Company}";
    }
}

// Производный класс 3 (Сложное наследование)
public class VIPCorporateCard : CorporateCard
{
    public string PersonalManager { get; set; }

    public VIPCorporateCard(string cardNumber, string holderName, string expiryDate, decimal initialBalance, string company, string personalManager) 
        : base(cardNumber, holderName, expiryDate, initialBalance, company)
    {
        PersonalManager = personalManager;
    }

    public void CallManager()
    {
        Console.WriteLine($"[VIP-Сервис] Осуществляется звонок вашему персональному менеджеру: {PersonalManager}...");
    }
}

Карта: 1111-2222-3333 | Владелец: Иван Иванов | Срок: 12.28
Текущий баланс: 50000 руб.

Карта: 7777-8888-9999 | Владелец: Петр Петров | Срок: 10.29 | Компания: ООО Инновации
Текущий баланс: 500000 руб.

Перевод 25000 руб. с карты 7777-8888-9999 на карту 1111-2222-3333 выполнен успешно.

Текущий баланс: 75000 руб.
Текущий баланс: 475000 руб.
--------------------------------------------------

[Демонстрация дополнительного задания]
Карта: 5555-6666-7777 | Владелец: Анна Смирнова | Срок: 11.30 | Компания: ООО Инновации
[VIP-Сервис] Осуществляется звонок вашему персональному менеджеру: Михаил...
Оплата 5000 руб. успешно проведена.
[Бонус] Начислено 50 баллов. Баланс баллов: 50.
[Безопасность] Карта 7777-8888-9999 была заблокирована.
[Отказ] Карта 7777-8888-9999 заблокирована.


--- Перегрузка метода Pay ---
Оплата 3000 руб. в категории 'Рестораны' успешно проведена.
[Бонус] Начислено 30 баллов. Баланс баллов: 80.
 -> [Кэшбэк] Применен повышенный кэшбэк для категории 'Рестораны'!
Оплата 150